# Step 5H ? Paired turn-boundary CNN retraining for v7

Step 5F showed that forward-only hard negatives reduce some premature right-turn predictions but can suppress legitimate turn recognition. Step 5G therefore collected matched human-command boundary states: stable forward frames immediately before each turn onset and true left/right frames immediately after onset.

This notebook changes the training data only. It adds paired boundary samples from two training sessions while keeping the CNN architecture, 50?50 RGB preprocessing, optimizer, random seed, early-stopping policy, recovery_02 validation session, recovery_04 repeat reference, and class-weight alpha 0.50 fixed.

A third boundary session is packaged separately and never used by model.fit() or early stopping. It is evaluated only after the model is fixed.


## Locked data roles and pre-committed offline gate

Main model split:
- Train: original_01, original_02, original_03, recovery_01, recovery_03, boundary_train_01, boundary_train_02
- Training-time validation: recovery_02
- Repeat reference: recovery_04

Boundary-specific holdout:
- boundary_validation_01 ? never used for training or early stopping

The Step 5F forward-only hard-negative session is intentionally excluded so the effect of paired boundary data is isolated.

Offline promotion gate fixed before v7 training:
- recovery_02 forward recall >= 95%;
- recovery_02 left recall >= 70%;
- recovery_02 right recall >= 70%;
- boundary holdout pre-turn forward accuracy >= 70%;
- boundary holdout post-turn left recall >= 70%;
- boundary holdout post-turn right recall >= 70%;
- at least 60% of boundary events must have a majority-correct pre-turn side AND a majority-correct post-turn side.

Passing this gate permits a fresh human-controlled no-display shadow traversal only. It does not permit autonomous motor control.


In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intelligent_Robotics_Design_Project"
SESSION_DATASET_ZIP_NAME = "step05h_paired_boundary_dataset_v1.zip"
OUTPUT_MODEL_NAME = "maze_navigation_model_v7.h5"
OUTPUT_METRICS_NAME = "step05h_v7_metrics.json"
OUTPUT_CONFUSION_NAME = "step05h_v7_reference_confusion.csv"
OUTPUT_BOUNDARY_CONFUSION_NAME = "step05h_v7_boundary_validation_confusion.csv"
OUTPUT_BOUNDARY_EVENTS_NAME = "step05h_v7_boundary_event_metrics.csv"
OUTPUT_HISTORY_NAME = "step05h_v7_history.csv"

RANDOM_SEED = 42
IMAGE_SIZE = (50, 50)
BATCH_SIZE = 32
EPOCHS = 30
CLASS_WEIGHT_ALPHA = 0.50

CLASS_NAMES = ("forward", "left", "right")
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

EXPECTED_SPLIT_SESSIONS = {
    "train": {
        "original_01",
        "original_02",
        "original_03",
        "recovery_01",
        "recovery_03",
        "boundary_train_01",
        "boundary_train_02",
    },
    "validation": {"recovery_02"},
    "test": {"recovery_04"},
}
EXPECTED_BOUNDARY_VALIDATION_SESSION = "boundary_validation_01"

RECOVERY_FORWARD_RECALL_FLOOR = 0.95
RECOVERY_TURN_RECALL_FLOOR = 0.70
BOUNDARY_PRE_FORWARD_FLOOR = 0.70
BOUNDARY_TURN_RECALL_FLOOR = 0.70
BOUNDARY_EVENT_PASS_FLOOR = 0.60

ALPHA050_BOUNDARY_BASELINE = {
    "pre_forward_accuracy": 0.34,
    "post_turn_accuracy": 0.74,
    "forward_recall": 0.34,
    "left_recall": 0.65,
    "right_recall": 0.80,
    "event_pass_rate": 0.10,
}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()

project_dir = Path(DRIVE_PROJECT_DIR)
zip_path = project_dir / SESSION_DATASET_ZIP_NAME
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the Step 5H paired-boundary dataset ZIP here first: {zip_path}")

print(f"TensorFlow: {tf.__version__}")
print(f"Dataset ZIP: {zip_path}")


In [ ]:
extract_root = Path("/content/step05h_paired_boundary_dataset")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

manifest_path = extract_root / "manifest.csv"
split_plan_path = extract_root / "split_plan.csv"
boundary_validation_path = extract_root / "boundary_validation_manifest.csv"
paired_training_path = extract_root / "paired_boundary_training_manifest.csv"

required_files = [
    manifest_path,
    split_plan_path,
    boundary_validation_path,
    paired_training_path,
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Dataset package is missing: {missing_files}")

manifest = pd.read_csv(manifest_path)
required_columns = {"session_id", "split", "label", "relative_path", "source_filename"}
if set(manifest.columns) != required_columns:
    raise ValueError(f"Unexpected manifest columns: {list(manifest.columns)}")

if not set(manifest["label"]).issubset(CLASS_TO_INDEX):
    raise ValueError("Manifest contains an unknown class label")

manifest["label_index"] = manifest["label"].map(CLASS_TO_INDEX).astype(np.int32)
manifest["image_path"] = manifest["relative_path"].map(lambda value: str(extract_root / value))

missing = [path for path in manifest["image_path"] if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(f"Missing extracted images, first example: {missing[0]}")

for split, expected_sessions in EXPECTED_SPLIT_SESSIONS.items():
    actual_sessions = set(manifest.loc[manifest["split"] == split, "session_id"])
    if actual_sessions != expected_sessions:
        raise ValueError(
            f"{split} session mismatch: expected {sorted(expected_sessions)}, got {sorted(actual_sessions)}"
        )

all_session_sets = list(EXPECTED_SPLIT_SESSIONS.values())
for i in range(len(all_session_sets)):
    for j in range(i + 1, len(all_session_sets)):
        if not all_session_sets[i].isdisjoint(all_session_sets[j]):
            raise ValueError("A session appears in more than one main split")

if EXPECTED_BOUNDARY_VALIDATION_SESSION in set(manifest["session_id"]):
    raise ValueError("Boundary validation leaked into the main model manifest")

print("MAIN SESSION ISOLATION CHECK: PASS")
print(f"Main images: {len(manifest)}")


In [ ]:
paired_training = pd.read_csv(paired_training_path)
boundary_validation = pd.read_csv(boundary_validation_path)

expected_boundary_columns = {
    "session_id",
    "event_id",
    "turn_class",
    "boundary_side",
    "relative_offset",
    "frame_index",
    "label",
    "relative_path",
    "robot_x",
    "robot_y",
    "robot_yaw",
}
if set(paired_training.columns) != expected_boundary_columns:
    raise ValueError("Unexpected paired training manifest columns")
if set(boundary_validation.columns) != expected_boundary_columns:
    raise ValueError("Unexpected boundary validation manifest columns")

if set(paired_training["session_id"]) != {"boundary_train_01", "boundary_train_02"}:
    raise ValueError("Paired training manifest contains unexpected sessions")
if set(boundary_validation["session_id"]) != {EXPECTED_BOUNDARY_VALIDATION_SESSION}:
    raise ValueError("Boundary validation manifest contains an unexpected session")

if len(paired_training) != 180:
    raise ValueError(f"Expected 180 paired training images, got {len(paired_training)}")
if len(boundary_validation) != 100:
    raise ValueError(f"Expected 100 held-out boundary images, got {len(boundary_validation)}")

expected_train_labels = {"forward": 90, "left": 45, "right": 45}
actual_train_labels = paired_training["label"].value_counts().to_dict()
if actual_train_labels != expected_train_labels:
    raise ValueError(f"Unexpected paired training label counts: {actual_train_labels}")

expected_boundary_labels = {"forward": 50, "left": 20, "right": 30}
actual_boundary_labels = boundary_validation["label"].value_counts().to_dict()
if actual_boundary_labels != expected_boundary_labels:
    raise ValueError(f"Unexpected boundary validation label counts: {actual_boundary_labels}")

for frame in (paired_training, boundary_validation):
    frame["label_index"] = frame["label"].map(CLASS_TO_INDEX).astype(np.int32)
    frame["image_path"] = frame["relative_path"].map(lambda value: str(extract_root / value))
    missing = [path for path in frame["image_path"] if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing boundary image: {missing[0]}")

# Every boundary event must retain 5 pre-forward + 5 post-turn images.
for name, frame in [
    ("paired training", paired_training),
    ("boundary validation", boundary_validation),
]:
    for (session_id, event_id), group in frame.groupby(["session_id", "event_id"]):
        if len(group) != 10:
            raise ValueError(f"{name} {session_id} event {event_id}: expected 10 rows")
        if (group["boundary_side"] == "pre_forward").sum() != 5:
            raise ValueError(f"{name} {session_id} event {event_id}: expected 5 pre rows")
        if (group["boundary_side"] == "post_turn").sum() != 5:
            raise ValueError(f"{name} {session_id} event {event_id}: expected 5 post rows")

print("PAIRED BOUNDARY CONTRACT: PASS")
print("Paired train labels:", actual_train_labels)
print("Held-out boundary labels:", actual_boundary_labels)


In [ ]:
summary = (
    manifest.groupby(["split", "session_id", "label"])
    .size()
    .rename("images")
    .reset_index()
)
display(summary)

split_counts = (
    manifest.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_NAMES, fill_value=0)
)
display(split_counts)

for split in ("train", "validation", "test"):
    frame = manifest[manifest["split"] == split]
    print(f"{split}: {len(frame)} images | sessions={sorted(frame['session_id'].unique().tolist())}")

print(
    "Boundary holdout:",
    len(boundary_validation),
    "images | events=",
    boundary_validation["event_id"].nunique(),
    "| session=",
    EXPECTED_BOUNDARY_VALIDATION_SESSION,
)


In [ ]:
def load_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32) / 255.0

def load_example(path, label):
    return load_image(path), label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame["image_path"].to_numpy(), frame["label_index"].to_numpy(dtype=np.int32))
    )
    dataset = dataset.map(load_example, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    if training:
        dataset = dataset.shuffle(len(frame), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)
boundary_df = boundary_validation.reset_index(drop=True)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)
boundary_ds = make_dataset(boundary_df, training=False)

print("Datasets ready.")
print("Boundary holdout is evaluation-only and is not passed to model.fit().")


In [ ]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model()
model.summary()


In [ ]:
train_labels = train_df["label_index"].to_numpy(dtype=np.int32)
train_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))

balanced = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=train_labels,
).astype(np.float64)

raw_weights = np.power(balanced, CLASS_WEIGHT_ALPHA)
mean_sample_weight = float(np.dot(train_counts, raw_weights) / train_counts.sum())
normalized = raw_weights / mean_sample_weight
class_weights = {index: float(normalized[index]) for index in range(len(CLASS_NAMES))}

print("Fixed class-weight alpha:", CLASS_WEIGHT_ALPHA)
print("Training counts:", {CLASS_NAMES[i]: int(train_counts[i]) for i in range(3)})
print("Class weights:", {CLASS_NAMES[i]: class_weights[i] for i in range(3)})


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=2,
)

history_df = pd.DataFrame(history.history)
print(f"Epochs completed: {len(history_df)}")
display(history_df.tail())

plt.figure(figsize=(8, 4))
plt.plot(history_df.index + 1, history_df["loss"], label="train loss")
plt.plot(history_df.index + 1, history_df["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
def prediction_metrics(frame, probabilities):
    labels = frame["label_index"].to_numpy(dtype=np.int32)
    predictions = np.argmax(probabilities, axis=1)
    confusion = confusion_matrix(labels, predictions, labels=np.arange(3))
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=np.arange(3), zero_division=0
    )
    forward_index = CLASS_TO_INDEX["forward"]
    left_index = CLASS_TO_INDEX["left"]
    right_index = CLASS_TO_INDEX["right"]
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(np.mean(f1)),
        "precision": {CLASS_NAMES[i]: float(precision[i]) for i in range(3)},
        "recall": {CLASS_NAMES[i]: float(recall[i]) for i in range(3)},
        "f1": {CLASS_NAMES[i]: float(f1[i]) for i in range(3)},
        "support": {CLASS_NAMES[i]: int(support[i]) for i in range(3)},
        "forward_to_left": int(confusion[forward_index, left_index]),
        "forward_to_right": int(confusion[forward_index, right_index]),
        "forward_support": int(confusion[forward_index].sum()),
        "confusion_matrix": confusion,
    }

val_probabilities = model.predict(val_ds, verbose=0)
val_metrics = prediction_metrics(val_df, val_probabilities)
val_confusion = val_metrics.pop("confusion_matrix")

print(f"Validation accuracy: {val_metrics['accuracy']:.2%}")
print(f"Validation macro-F1: {val_metrics['macro_f1']:.4f}")
print("Validation recall:", {k: f"{v:.2%}" for k, v in val_metrics["recall"].items()})
print(
    "Validation forward->left/right:",
    val_metrics["forward_to_left"],
    val_metrics["forward_to_right"],
    "of",
    val_metrics["forward_support"],
)
display(pd.DataFrame(val_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))


## Repeat-reference evaluation

The model is now fixed. recovery_04 is evaluated below as a repeat reference after training and recovery_02 validation. It has been inspected in earlier experiments, so it is not described as a pristine final holdout.

The independent Step 5G boundary validation session is evaluated separately afterward.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
test_probabilities = model.predict(test_ds, verbose=0)
test_metrics = prediction_metrics(test_df, test_probabilities)
test_confusion = test_metrics.pop("confusion_matrix")

print(f"TEST loss: {test_loss:.4f}")
print(f"TEST accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"TEST macro-F1: {test_metrics['macro_f1']:.4f}")
print("TEST recall:", {k: f"{v:.2%}" for k, v in test_metrics["recall"].items()})
print(
    "TEST forward->left/right:",
    test_metrics["forward_to_left"],
    test_metrics["forward_to_right"],
    "of",
    test_metrics["forward_support"],
)
display(pd.DataFrame(test_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))


In [ ]:
boundary_probabilities = model.predict(boundary_ds, verbose=0)
boundary_metrics = prediction_metrics(boundary_df, boundary_probabilities)
boundary_confusion = boundary_metrics.pop("confusion_matrix")
boundary_predictions = np.argmax(boundary_probabilities, axis=1)
boundary_labels = boundary_df["label_index"].to_numpy(dtype=np.int32)

pre_mask = boundary_df["boundary_side"].eq("pre_forward").to_numpy()
post_mask = boundary_df["boundary_side"].eq("post_turn").to_numpy()

pre_forward_accuracy = float(
    accuracy_score(boundary_labels[pre_mask], boundary_predictions[pre_mask])
)
post_turn_accuracy = float(
    accuracy_score(boundary_labels[post_mask], boundary_predictions[post_mask])
)

left_index = CLASS_TO_INDEX["left"]
right_index = CLASS_TO_INDEX["right"]
pre_false_left = int(np.sum(boundary_predictions[pre_mask] == left_index))
pre_false_right = int(np.sum(boundary_predictions[pre_mask] == right_index))

event_rows = []
for event_id, group in boundary_df.groupby("event_id", sort=True):
    indices = group.index.to_numpy(dtype=np.int32)
    pre_indices = group.index[group["boundary_side"] == "pre_forward"].to_numpy(dtype=np.int32)
    post_indices = group.index[group["boundary_side"] == "post_turn"].to_numpy(dtype=np.int32)
    turn_class = str(group.loc[post_indices[0], "turn_class"])
    turn_index = CLASS_TO_INDEX[turn_class]

    pre_correct = int(np.sum(boundary_predictions[pre_indices] == CLASS_TO_INDEX["forward"]))
    post_correct = int(np.sum(boundary_predictions[post_indices] == turn_index))
    event_pass = bool(pre_correct >= 3 and post_correct >= 3)

    event_rows.append(
        {
            "event_id": int(event_id),
            "turn_class": turn_class,
            "pre_forward_correct_of_5": pre_correct,
            "post_turn_correct_of_5": post_correct,
            "event_majority_pass": event_pass,
        }
    )

boundary_event_df = pd.DataFrame(event_rows)
event_pass_rate = float(boundary_event_df["event_majority_pass"].mean())

boundary_detail = {
    "pre_forward_accuracy": pre_forward_accuracy,
    "post_turn_accuracy": post_turn_accuracy,
    "pre_forward_false_left": pre_false_left,
    "pre_forward_false_right": pre_false_right,
    "event_pass_rate": event_pass_rate,
    "events_passed": int(boundary_event_df["event_majority_pass"].sum()),
    "events_total": int(len(boundary_event_df)),
}

offline_gate_checks = {
    "recovery_forward_recall": bool(
        val_metrics["recall"]["forward"] >= RECOVERY_FORWARD_RECALL_FLOOR
    ),
    "recovery_left_recall": bool(
        val_metrics["recall"]["left"] >= RECOVERY_TURN_RECALL_FLOOR
    ),
    "recovery_right_recall": bool(
        val_metrics["recall"]["right"] >= RECOVERY_TURN_RECALL_FLOOR
    ),
    "boundary_pre_forward_accuracy": bool(
        pre_forward_accuracy >= BOUNDARY_PRE_FORWARD_FLOOR
    ),
    "boundary_left_recall": bool(
        boundary_metrics["recall"]["left"] >= BOUNDARY_TURN_RECALL_FLOOR
    ),
    "boundary_right_recall": bool(
        boundary_metrics["recall"]["right"] >= BOUNDARY_TURN_RECALL_FLOOR
    ),
    "boundary_event_pass_rate": bool(
        event_pass_rate >= BOUNDARY_EVENT_PASS_FLOOR
    ),
}
offline_gate_pass = bool(all(offline_gate_checks.values()))

print(f"Boundary pre-turn forward accuracy: {pre_forward_accuracy:.2%}")
print(f"Boundary post-turn accuracy: {post_turn_accuracy:.2%}")
print(
    "Boundary recall:",
    {k: f"{v:.2%}" for k, v in boundary_metrics["recall"].items()},
)
print(
    "Boundary pre-turn false-left/right:",
    pre_false_left,
    pre_false_right,
    "of",
    int(pre_mask.sum()),
)
print(
    f"Boundary event majority pass: {boundary_detail['events_passed']}/"
    f"{boundary_detail['events_total']} ({event_pass_rate:.2%})"
)
print("Frozen alpha-0.50 boundary baseline:", ALPHA050_BOUNDARY_BASELINE)
print("OFFLINE GATE CHECKS:", offline_gate_checks)
print("OFFLINE GATE:", "PASS FOR LIVE SHADOW ONLY" if offline_gate_pass else "FAIL")
display(pd.DataFrame(boundary_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))
display(boundary_event_df)


In [ ]:
project_dir.mkdir(parents=True, exist_ok=True)

model_path = project_dir / OUTPUT_MODEL_NAME
metrics_path = project_dir / OUTPUT_METRICS_NAME
confusion_path = project_dir / OUTPUT_CONFUSION_NAME
boundary_confusion_path = project_dir / OUTPUT_BOUNDARY_CONFUSION_NAME
boundary_events_path = project_dir / OUTPUT_BOUNDARY_EVENTS_NAME
history_path = project_dir / OUTPUT_HISTORY_NAME

model.save(model_path)
pd.DataFrame(test_confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(confusion_path)
pd.DataFrame(boundary_confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    boundary_confusion_path
)
boundary_event_df.to_csv(boundary_events_path, index=False)
history_df.to_csv(history_path, index=False)

metrics_payload = {
    "method": "session-aware paired turn-boundary retraining",
    "random_seed": RANDOM_SEED,
    "image_size": list(IMAGE_SIZE),
    "class_weight_alpha": CLASS_WEIGHT_ALPHA,
    "split_sessions": {key: sorted(value) for key, value in EXPECTED_SPLIT_SESSIONS.items()},
    "boundary_validation_session": EXPECTED_BOUNDARY_VALIDATION_SESSION,
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df),
    "boundary_validation_images": len(boundary_df),
    "epochs_completed": len(history_df),
    "validation": val_metrics,
    "test": test_metrics,
    "boundary_validation": {
        **boundary_metrics,
        **boundary_detail,
    },
    "offline_gate_thresholds": {
        "recovery_forward_recall_floor": RECOVERY_FORWARD_RECALL_FLOOR,
        "recovery_turn_recall_floor": RECOVERY_TURN_RECALL_FLOOR,
        "boundary_pre_forward_floor": BOUNDARY_PRE_FORWARD_FLOOR,
        "boundary_turn_recall_floor": BOUNDARY_TURN_RECALL_FLOOR,
        "boundary_event_pass_floor": BOUNDARY_EVENT_PASS_FLOOR,
    },
    "offline_gate_checks": offline_gate_checks,
    "offline_gate_pass": offline_gate_pass,
    "alpha050_boundary_baseline": ALPHA050_BOUNDARY_BASELINE,
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

print(f"Saved v7 model: {model_path}")
print(f"Saved metrics: {metrics_path}")
print(f"Saved repeat-reference confusion: {confusion_path}")
print(f"Saved boundary confusion: {boundary_confusion_path}")
print(f"Saved boundary event metrics: {boundary_events_path}")
print(f"Saved training history: {history_path}")


## Step-6 gate after v7

Do not move directly to autonomous motor control from this notebook.

If the pre-committed offline gate prints **PASS FOR LIVE SHADOW ONLY**, place maze_navigation_model_v7.h5 in the local models folder, verify its contract locally, and run a fresh human-controlled no-display shadow traversal.

If the offline gate fails, stop and analyze the failed checks before any further live experiment.

Even an offline pass is only eligibility for live shadow validation. Autonomous Step 6 remains blocked until fresh sequential shadow evidence is satisfactory.
